# 🧠 Notebook 03b — EMA sobre embeddings Dialog2Flow · `version_4`

Esta notebook genera representaciones EMA sobre el espacio `dialog2flow-joint-bert-base`.

La actualización usada es:

```text
h_actual = LayerNorm((1 - alpha) * h_anterior + alpha * e_actual)
```

Se prueban valores de `alpha` entre 0.1 y 0.9 con saltos de 0.1, siguiendo la sugerencia de explorar cuánto peso conviene darle al turno actual frente al historial conversacional.

La notebook **no modifica** las representaciones acumulativas usadas en el paper. Genera nuevos archivos en `data/version_4/embeddings/`.

## 1️⃣ Instalación e imports

In [ ]:
!pip install -q gdown faiss-cpu pandas numpy tqdm

In [ ]:
import gc
import json
import time
import shutil
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from IPython.display import display

## 2️⃣ Montaje de Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3️⃣ Configuración

In [ ]:
# ============================================================
# Estructura versionada
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/ANN/TF")

SOURCE_VERSION = "version_3"
VERSION = "version_4"

SOURCE_DATA_DIR = BASE_DIR / "data" / SOURCE_VERSION
SOURCE_EMBEDDINGS_DIR = SOURCE_DATA_DIR / "embeddings"

DATA_DIR = BASE_DIR / "data" / VERSION
RESULTS_DIR = BASE_DIR / "resultados" / VERSION

EMBEDDINGS_DIR = DATA_DIR / "embeddings"
SPLITS_DIR = DATA_DIR / "splits"

GEOMETRY_RESULTS_DIR = RESULTS_DIR / "geometry"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in [DATA_DIR, EMBEDDINGS_DIR, SPLITS_DIR, GEOMETRY_RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# Experimento EMA
# ============================================================

SHORT_NAME = "dialog2flow"
EMBEDDING_LABEL = "Dialog2Flow"

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

RANDOM_STATE = 42
N_QUERIES = 10_000
K_MAX = 100
N_MUESTRA_ANALISIS = 100_000

# Si RUN_EXACT_OVERLAP=True se calcula Overlap@1/10/100 entre vecinos exactos
# de la representación estática y cada EMA. Es costoso, pero útil.
RUN_EXACT_OVERLAP = True

print("BASE_DIR:", BASE_DIR)
print("SOURCE_DATA_DIR:", SOURCE_DATA_DIR)
print("DATA_DIR:", DATA_DIR)
print("ALPHA_VALUES:", ALPHA_VALUES)

## 4️⃣ Carga de insumos

In [ ]:
def find_existing_file(filename: str, candidates):
    for base in candidates:
        p = base / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"No se encontró {filename}. Busqué en: " + ", ".join(str(c) for c in candidates)
    )


# Se prioriza version_3/embeddings, pero se permite fallback a version_3 plano.
static_path = find_existing_file(
    "embeddings_dialog2flow.npy",
    [SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

dialogs_path = find_existing_file(
    "dialogs-2.0.pkl",
    [SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

ids_path = find_existing_file(
    "ids.npy",
    [SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

embeddings = np.load(static_path, mmap_mode="r")
df = pd.read_pickle(dialogs_path)
ids = np.load(ids_path)

print("Embeddings:", embeddings.shape, embeddings.dtype)
print("DataFrame:", df.shape)
print("IDs:", ids.shape)

# Copiamos metadatos mínimos a version_4 para dejar la carpeta autosuficiente.
for src in [dialogs_path, ids_path, static_path]:
    dst = EMBEDDINGS_DIR / src.name if src.suffix == ".npy" else DATA_DIR / src.name
    if not dst.exists():
        print("Copiando a version_4:", src.name)
        shutil.copy(src, dst)

display(df.head())

## 5️⃣ Funciones auxiliares

In [ ]:
def alpha_tag(alpha: float) -> str:
    return f"{alpha:.1f}".replace(".", "_")


def ema_embedding_filename(alpha: float) -> str:
    return f"ema_embeddings_dialog2flow_alpha_{alpha_tag(alpha)}.npy"


def ema_similarity_filename(alpha: float) -> str:
    return f"similitud_static_ema_dialog2flow_alpha_{alpha_tag(alpha)}.npy"


def turn_position_filename(alpha: float) -> str:
    return f"turn_position_ema_dialog2flow_alpha_{alpha_tag(alpha)}.npy"


def layer_norm_np(x: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    x = np.asarray(x, dtype="float32")
    mean = x.mean()
    var = x.var()
    return ((x - mean) / np.sqrt(var + eps)).astype("float32")


def cosine_np(a: np.ndarray, b: np.ndarray, eps: float = 1e-12) -> float:
    return float(np.dot(a, b) / ((np.linalg.norm(a) + eps) * (np.linalg.norm(b) + eps)))


def preparar_split(n_total: int, n_queries: int, random_state: int):
    split_index_path = SPLITS_DIR / f"index_idx_N{n_queries}_seed{random_state}.npy"
    split_query_path = SPLITS_DIR / f"query_idx_N{n_queries}_seed{random_state}.npy"

    # Si existen en version_4, se usan.
    if split_index_path.exists() and split_query_path.exists():
        return np.load(split_index_path), np.load(split_query_path)

    # Si existen en version_3, se copian.
    src_split_dir = SOURCE_DATA_DIR / "splits"
    src_index = src_split_dir / split_index_path.name
    src_query = src_split_dir / split_query_path.name
    if src_index.exists() and src_query.exists():
        shutil.copy(src_index, split_index_path)
        shutil.copy(src_query, split_query_path)
        return np.load(split_index_path), np.load(split_query_path)

    # Si no existen, se generan con default_rng(42), como en la notebook persistente.
    rng = np.random.default_rng(random_state)
    query_idx = rng.choice(n_total, size=n_queries, replace=False).astype("int64")
    mask = np.ones(n_total, dtype=bool)
    mask[query_idx] = False
    index_idx = np.where(mask)[0].astype("int64")

    np.save(split_index_path, index_idx)
    np.save(split_query_path, query_idx)
    return index_idx, query_idx

## 6️⃣ Generación de embeddings EMA

In [ ]:
def generar_ema_embeddings(alpha: float):
    out_path = EMBEDDINGS_DIR / ema_embedding_filename(alpha)
    sim_path = GEOMETRY_RESULTS_DIR / ema_similarity_filename(alpha)
    turn_path = GEOMETRY_RESULTS_DIR / turn_position_filename(alpha)

    n, d = embeddings.shape

    if out_path.exists() and sim_path.exists() and turn_path.exists():
        print(f"Alpha={alpha}: archivos existentes. Se omite generación.")
        return out_path, sim_path, turn_path

    print("=" * 100)
    print(f"Generando EMA alpha={alpha}")
    print("Salida:", out_path)

    ema = np.lib.format.open_memmap(out_path, mode="w+", dtype="float32", shape=(n, d))
    sim = np.lib.format.open_memmap(sim_path, mode="w+", dtype="float32", shape=(n,))
    turn_pos = np.lib.format.open_memmap(turn_path, mode="w+", dtype="int32", shape=(n,))

    t0 = time.time()

    for i, (dialogue_id, group) in enumerate(tqdm(df.groupby("dialogue_id", sort=False))):
        group = group.sort_values("turn_id")
        h = np.zeros(d, dtype="float32")

        for local_pos, row_id in enumerate(group.index.to_numpy()):
            e = np.asarray(embeddings[row_id], dtype="float32")
            h = layer_norm_np((1.0 - alpha) * h + alpha * e)
            ema[row_id] = h
            sim[row_id] = cosine_np(e, h)
            turn_pos[row_id] = local_pos

        if (i + 1) % 10000 == 0:
            ema.flush()
            sim.flush()
            turn_pos.flush()

    ema.flush()
    sim.flush()
    turn_pos.flush()

    print(f"Alpha={alpha} finalizado en {time.time() - t0:.2f} segundos")
    return out_path, sim_path, turn_path


ema_files = []
for alpha in ALPHA_VALUES:
    ema_files.append((alpha, *generar_ema_embeddings(alpha)))

ema_files

## 7️⃣ Métricas geométricas básicas

In [ ]:
rows = []

rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(N_MUESTRA_ANALISIS, embeddings.shape[0])
sample_idx = rng.choice(embeddings.shape[0], size=sample_size, replace=False)

for alpha, emb_path, sim_path, turn_path in ema_files:
    sim = np.load(sim_path, mmap_mode="r")
    turn_pos = np.load(turn_path, mmap_mode="r")

    sim_sample = np.asarray(sim[sample_idx])
    turn_sample = np.asarray(turn_pos[sample_idx])

    rows.append({
        "embedding_base": EMBEDDING_LABEL,
        "variant": f"ema_alpha_{alpha_tag(alpha)}",
        "alpha": alpha,
        "sample_size": sample_size,
        "similarity_mean": float(sim_sample.mean()),
        "similarity_median": float(np.median(sim_sample)),
        "similarity_min": float(sim_sample.min()),
        "similarity_max": float(sim_sample.max()),
        "turn_position_mean": float(turn_sample.mean()),
        "turn_position_median": float(np.median(turn_sample)),
    })

df_geometry_basic = pd.DataFrame(rows)
display(df_geometry_basic)

timestamp = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y%m%d_%H%M%S_ARG")
basic_path = GEOMETRY_RESULTS_DIR / f"geometry_ema_basic_dialog2flow_{timestamp}.csv"
latest_basic_path = GEOMETRY_RESULTS_DIR / "geometry_ema_basic_dialog2flow_latest.csv"

df_geometry_basic.to_csv(basic_path, index=False)
df_geometry_basic.to_csv(latest_basic_path, index=False)

print("Guardado:", basic_path)
print("Latest:", latest_basic_path)

## 8️⃣ Overlap exacto estático vs EMA

In [ ]:
def recall_at_k(I_a, I_b, k: int) -> float:
    vals = []
    for a, b in zip(I_a[:, :k], I_b[:, :k]):
        vals.append(len(set(a).intersection(set(b))) / k)
    return float(np.mean(vals))


def obtener_vecinos_flat(matriz, index_idx, query_idx, k):
    index_vectors = np.asarray(matriz[index_idx], dtype="float32")
    query_vectors = np.asarray(matriz[query_idx], dtype="float32")

    faiss.normalize_L2(index_vectors)
    faiss.normalize_L2(query_vectors)

    d = index_vectors.shape[1]
    index_flat = faiss.IndexFlatL2(d)
    index_flat.add(index_vectors)

    D, I = index_flat.search(query_vectors, k)

    del index_flat, D, index_vectors, query_vectors
    gc.collect()

    return I


if RUN_EXACT_OVERLAP:
    index_idx, query_idx = preparar_split(
        n_total=embeddings.shape[0],
        n_queries=N_QUERIES,
        random_state=RANDOM_STATE,
    )

    print("Calculando vecinos exactos de la representación estática...")
    I_static_exact = obtener_vecinos_flat(embeddings, index_idx, query_idx, K_MAX)

    rows = []
    for alpha, emb_path, sim_path, turn_path in ema_files:
        print("=" * 100)
        print("Alpha:", alpha)

        ema = np.load(emb_path, mmap_mode="r")
        I_ema_exact = obtener_vecinos_flat(ema, index_idx, query_idx, K_MAX)

        rows.append({
            "embedding_base": EMBEDDING_LABEL,
            "variant": f"ema_alpha_{alpha_tag(alpha)}",
            "alpha": alpha,
            "overlap@1": recall_at_k(I_static_exact, I_ema_exact, 1),
            "overlap@10": recall_at_k(I_static_exact, I_ema_exact, 10),
            "overlap@100": recall_at_k(I_static_exact, I_ema_exact, 100),
        })

        del I_ema_exact, ema
        gc.collect()

    df_overlap = pd.DataFrame(rows)
    display(df_overlap)

    overlap_path = GEOMETRY_RESULTS_DIR / f"geometry_ema_overlap_dialog2flow_{timestamp}.csv"
    latest_overlap_path = GEOMETRY_RESULTS_DIR / "geometry_ema_overlap_dialog2flow_latest.csv"

    df_overlap.to_csv(overlap_path, index=False)
    df_overlap.to_csv(latest_overlap_path, index=False)

    print("Guardado:", overlap_path)
    print("Latest:", latest_overlap_path)
else:
    print("RUN_EXACT_OVERLAP=False. Se omite cálculo de overlap exacto.")

## 9️⃣ Vista final

In [ ]:
print("Archivos EMA generados:")
for alpha, emb_path, sim_path, turn_path in ema_files:
    print(alpha, "->", emb_path.name)